# Fake Product Review Detection
## Machine Learning Course Project

**Objective:** Build a binary classifier that detects whether a product review is **Fake (CG — Computer Generated)** or **Real (OR — Original)**.

**Dataset:** Amazon Fake Reviews Dataset (Kaggle — mexwell/fake-reviews-dataset)

**Models compared:**
- Logistic Regression
- Complement Naive Bayes
- Linear SVM
- Random Forest

**Features used:**
- TF-IDF (unigrams + bigrams, top 5000)
- Review length, word count, average word length
- Exclamation mark count
- VADER sentiment score
- Rating–sentiment gap

---

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# NLP
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)

warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# Create output dirs
os.makedirs('../visuals', exist_ok=True)
os.makedirs('../models',  exist_ok=True)

# Plot style — dark theme
plt.rcParams.update({
    'figure.facecolor': '#0a0b0f',
    'axes.facecolor':   '#111318',
    'axes.edgecolor':   '#2a2d3a',
    'axes.labelcolor':  '#e8eaf0',
    'text.color':       '#e8eaf0',
    'xtick.color':      '#7a7f9a',
    'ytick.color':      '#7a7f9a',
    'grid.color':       '#1a1d26',
    'figure.dpi':       120,
})

MODEL_COLORS = {
    'Logistic Regression': '#6c63ff',
    'Naive Bayes':         '#00d4aa',
    'SVM':                 '#f4a261',
    'Random Forest':       '#ff4d6d',
}

print('✅ All imports successful.')

## 2. Load Dataset

The dataset contains ~40,000 Amazon product reviews labelled as:
- **CG** (Computer Generated) → Fake
- **OR** (Original) → Real

In [ ]:
DATA_PATH = '../data/reviews.csv'

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]

# Ensure required columns exist
assert 'text_'  in df.columns, "Expected column 'text_'"
assert 'label'  in df.columns, "Expected column 'label'"

if 'rating' not in df.columns:
    df['rating'] = 3.0

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'\nLabel counts:\n{df["label"].value_counts()}')
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
# Preview sample rows
df.sample(5, random_state=42)[['label', 'rating', 'text_']]

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class Distribution

In [ ]:
counts = df['label'].value_counts()
label_map = {'CG': 'Fake (CG)', 'OR': 'Real (OR)'}

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    [label_map.get(l, l) for l in counts.index],
    counts.values,
    color=['#ff4d6d', '#00d4aa'], width=0.5, edgecolor='#2a2d3a'
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=11)

ax.set_title('Class Distribution — Fake vs Real Reviews', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Number of Reviews')
ax.grid(True, axis='y', alpha=0.15)
plt.tight_layout()
plt.savefig('../visuals/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Class balance: {(counts / counts.sum() * 100).round(1).to_dict()}')

### 3.2 Star Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, label, color, title in zip(
    axes,
    ['CG', 'OR'],
    ['#ff4d6d', '#00d4aa'],
    ['Fake Reviews — Rating Distribution', 'Real Reviews — Rating Distribution']
):
    subset = df[df['label'] == label]['rating'].dropna()
    ax.hist(subset, bins=[0.5,1.5,2.5,3.5,4.5,5.5], color=color,
            edgecolor='#2a2d3a', alpha=0.85, rwidth=0.8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Star Rating')
    ax.set_ylabel('Count')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(True, axis='y', alpha=0.15)

plt.suptitle('Star Rating Distributions by Label', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Review Length Distribution

In [ ]:
df['review_length_raw'] = df['text_'].fillna('').str.len()

fig, ax = plt.subplots(figsize=(8, 4))

for label, color, name in [('CG', '#ff4d6d', 'Fake'), ('OR', '#00d4aa', 'Real')]:
    subset = df[df['label'] == label]['review_length_raw']
    ax.hist(subset.clip(upper=2000), bins=60, color=color, alpha=0.55,
            label=name, edgecolor='none')

ax.set_title('Review Length Distribution (characters)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Character Count (clipped at 2000)')
ax.set_ylabel('Frequency')
ax.legend(framealpha=0.2)
ax.grid(True, axis='y', alpha=0.15)
plt.tight_layout()
plt.savefig('../visuals/review_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Mean review length:')
print(df.groupby('label')['review_length_raw'].mean().round(1))

## 4. Text Preprocessing

The pipeline:
1. Lowercase the text
2. Remove URLs and HTML tags
3. Strip punctuation
4. Tokenise by whitespace
5. Remove English stopwords
6. Lemmatize each remaining token

**Why lemmatize?** Reduces vocabulary size by collapsing inflected forms (e.g., *running* → *run*, *better* → *good*). This improves TF-IDF quality without losing semantic meaning.

In [ ]:
_STOP_WORDS  = set(stopwords.words('english'))
_LEMMATIZER  = WordNetLemmatizer()
_PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # URLs
    text = re.sub(r'<[^>]+>', '', text)              # HTML tags
    text = text.translate(_PUNCT_TABLE)              # Punctuation
    tokens = text.split()
    tokens = [
        _LEMMATIZER.lemmatize(tok)
        for tok in tokens
        if tok.isalpha() and tok not in _STOP_WORDS
    ]
    return ' '.join(tokens)

# Quick sanity check
sample = 'BEST PRODUCT EVER!!!! Absolutely amazing!! Will buy again!!!'
print(f'Original : {sample}')
print(f'Cleaned  : {clean_text(sample)}')

In [ ]:
print('Cleaning all reviews (may take 1–2 minutes)…')
df['cleaned_text'] = df['text_'].fillna('').apply(clean_text)
print(f'Done. Example cleaned review:\n\n{df["cleaned_text"].iloc[0][:300]}')

## 5. Feature Engineering

Beyond TF-IDF, we add numeric signals that capture **behavioural patterns** of fake reviews:

| Feature | Rationale |
|---|---|
| `review_length` | Fake reviews tend to be shorter or artificially padded |
| `word_count` | Complements character count |
| `avg_word_length` | Bot-generated text sometimes uses unusual word lengths |
| `exclamation_count` | Over-excited phrasing is a common fake review signal |
| `sentiment_score` | VADER compound score — extreme positivity is suspicious |
| `rating_sentiment_gap` | If a 5-star review has negative sentiment (or vice versa), it's suspicious |

In [ ]:
_ANALYZER = SentimentIntensityAnalyzer()

NUMERIC_FEATURES = [
    'review_length', 'word_count', 'avg_word_length',
    'exclamation_count', 'sentiment_score', 'rating_sentiment_gap'
]

raw = df['text_'].fillna('').astype(str)

df['review_length']     = raw.str.len()
df['word_count']        = raw.str.split().str.len().fillna(0)
df['avg_word_length']   = raw.apply(
    lambda t: float(np.mean([len(w) for w in t.split()])) if t.split() else 0.0
)
df['exclamation_count'] = raw.str.count('!')
df['sentiment_score']   = raw.apply(
    lambda t: _ANALYZER.polarity_scores(t)['compound']
)
rating = df['rating'].fillna(3.0).astype(float)
df['rating_sentiment_gap'] = (rating / 5.0 - (df['sentiment_score'] + 1.0) / 2.0).abs()

print('Feature statistics by label:')
df.groupby('label')[NUMERIC_FEATURES].mean().round(3)

### 5.1 Sentiment Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, feat, title in zip(
    axes,
    ['sentiment_score', 'rating_sentiment_gap'],
    ['Sentiment Score Distribution', 'Rating–Sentiment Gap Distribution']
):
    for label, color, name in [('CG', '#ff4d6d', 'Fake'), ('OR', '#00d4aa', 'Real')]:
        ax.hist(df[df['label'] == label][feat], bins=50, color=color,
                alpha=0.55, label=name, edgecolor='none', density=True)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(framealpha=0.2)
    ax.grid(True, axis='y', alpha=0.15)

plt.suptitle('Numeric Feature Distributions by Label', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Word Clouds

Word clouds give an intuitive view of the vocabulary most associated with **Fake** vs **Real** reviews.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, tag, cmap in [
    (axes[0], 'CG', 'Fake', 'Reds'),
    (axes[1], 'OR', 'Real', 'GnBu')
]:
    corpus = df[df['label'] == label]['cleaned_text'].dropna().str.cat(sep=' ')
    wc = WordCloud(
        width=800, height=400,
        background_color='#111318',
        colormap=cmap,
        max_words=150,
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Common Words — {tag} Reviews', fontsize=12,
                 fontweight='bold', color='#e8eaf0', pad=10)

plt.tight_layout()
plt.savefig('../visuals/wordclouds.png', dpi=150, bbox_inches='tight', facecolor='#0a0b0f')
plt.show()

## 7. Train / Test Split

We use a **stratified 80/20 split** to ensure both train and test sets have the same class ratio.

In [ ]:
# Binary label: 1 = Fake (CG), 0 = Real (OR)
df['binary_label'] = (df['label'].str.upper() == 'CG').astype(int)

feature_cols = ['cleaned_text'] + NUMERIC_FEATURES
X = df[feature_cols]
y = df['binary_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train set : {len(X_train):,} samples')
print(f'Test set  : {len(X_test):,} samples')
print(f'\nTrain class balance : {y_train.value_counts(normalize=True).round(3).to_dict()}')
print(f'Test  class balance : {y_test.value_counts(normalize=True).round(3).to_dict()}')

## 8. Build Sklearn Pipelines

Each pipeline is a two-stage sklearn `Pipeline`:
1. **Preprocessor** — `ColumnTransformer` that applies TF-IDF on `cleaned_text` and `StandardScaler` on numeric features
2. **Classifier** — the ML model

> **Note on Naive Bayes:** `MultinomialNB` requires non-negative features. Since `StandardScaler` can produce negative values, we use `ComplementNB` which handles this correctly and actually outperforms `MultinomialNB` on imbalanced datasets.

In [ ]:
# Preprocessor A: TF-IDF + StandardScaler (for LR, SVM, RF)
preprocessor_a = ColumnTransformer([
    ('tfidf',  TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True), 'cleaned_text'),
    ('scaler', StandardScaler(), NUMERIC_FEATURES),
])

# Preprocessor B: TF-IDF only (for Naive Bayes — avoids negative values)
preprocessor_b = ColumnTransformer([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True), 'cleaned_text'),
])

PIPELINES = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_a),
        ('classifier',   LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
    ]),
    'Naive Bayes': Pipeline([
        ('preprocessor', preprocessor_b),
        ('classifier',   ComplementNB(alpha=0.1)),
    ]),
    'SVM': Pipeline([
        ('preprocessor', preprocessor_a),
        # CalibratedClassifierCV wraps LinearSVC to enable predict_proba (needed for ROC curve)
        ('classifier',   CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=3000, random_state=42))),
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_a),
        ('classifier',   RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)),
    ]),
}

print('Pipelines ready:', list(PIPELINES.keys()))

## 9. Model Training & Evaluation

In [ ]:
all_metrics = {}
roc_data    = {}
best_f1     = -1
best_name   = None
best_pipe   = None

for name, pipe in PIPELINES.items():
    print(f'Training {name}…', end=' ', flush=True)
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    metrics = {
        'accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'f1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
    }
    all_metrics[name] = metrics
    roc_data[name]    = {'y_true': y_test.tolist(), 'y_prob': y_prob.tolist()}

    print(f'Acc={metrics["accuracy"]:.4f}  F1={metrics["f1"]:.4f}')

    if metrics['f1'] > best_f1:
        best_f1, best_name, best_pipe = metrics['f1'], name, pipe

print(f'\n🏆 Best model: {best_name} (F1 = {best_f1:.4f})')

### 9.1 Metrics Summary Table

In [ ]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df[['accuracy', 'precision', 'recall', 'f1']]
metrics_df.index.name = 'Model'
metrics_df.style.background_gradient(cmap='Purples', axis=0).format('{:.4f}')

### 9.2 Detailed Classification Report — Best Model

In [ ]:
y_pred_best = best_pipe.predict(X_test)
print(f'Classification Report — {best_name}\n')
print(classification_report(
    y_test, y_pred_best,
    target_names=['Real (OR)', 'Fake (CG)'],
    digits=4
))

## 10. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, pipe) in zip(axes, PIPELINES.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='RdPu',
        xticklabels=['Real', 'Fake'],
        yticklabels=['Real', 'Fake'],
        linewidths=0.5, linecolor='#2a2d3a',
        ax=ax,
    )
    f1 = all_metrics[name]['f1']
    ax.set_title(f'{name}  (F1={f1:.3f})', fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../visuals/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ROC Curves

The ROC (Receiver Operating Characteristic) curve shows the trade-off between **True Positive Rate** (sensitivity) and **False Positive Rate** for each model across all decision thresholds. A higher AUC (Area Under Curve) indicates a better model.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0,1], [0,1], 'w--', linewidth=1, alpha=0.4, label='Random Chance (AUC=0.50)')

for name, data in roc_data.items():
    fpr, tpr, _ = roc_curve(data['y_true'], data['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr,
            color=MODEL_COLORS.get(name, '#ffffff'),
            linewidth=2.2,
            label=f'{name}  (AUC = {roc_auc:.3f})')

ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
legend = ax.legend(loc='lower right', framealpha=0.15, fontsize=9)
plt.setp(legend.get_texts(), color='#e8eaf0')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig('../visuals/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Model Comparison Bar Chart

In [ ]:
names = list(all_metrics.keys())
accs  = [all_metrics[n]['accuracy']  for n in names]
precs = [all_metrics[n]['precision'] for n in names]
recs  = [all_metrics[n]['recall']    for n in names]
f1s   = [all_metrics[n]['f1']        for n in names]

x     = np.arange(len(names))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - 1.5*width, accs,  width, label='Accuracy',  color='#6c63ff', alpha=0.85)
b2 = ax.bar(x - 0.5*width, precs, width, label='Precision', color='#00d4aa', alpha=0.85)
b3 = ax.bar(x + 0.5*width, recs,  width, label='Recall',    color='#f4a261', alpha=0.85)
b4 = ax.bar(x + 1.5*width, f1s,   width, label='F1 Score',  color='#ff4d6d', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0.75, 1.00)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold', pad=12)
ax.legend(framealpha=0.2, fontsize=9)
ax.grid(True, axis='y', alpha=0.15)

for bars in [b1, b2, b3, b4]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../visuals/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Feature Importance (Random Forest)

Random Forest naturally ranks features by how much they reduce impurity across all trees. This gives us an interpretable view of which words and numeric signals are most predictive.

In [ ]:
rf_pipe  = PIPELINES['Random Forest']
rf_model = rf_pipe.named_steps['classifier']
rf_prep  = rf_pipe.named_steps['preprocessor']

tfidf_names = list(rf_prep.named_transformers_['tfidf'].get_feature_names_out())
all_names   = tfidf_names + NUMERIC_FEATURES

importances = rf_model.feature_importances_
top_n       = 20
indices     = np.argsort(importances)[-top_n:][::-1]

top_names  = [all_names[i] for i in indices]
top_values = importances[indices]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(
    range(top_n), top_values[::-1],
    color='#6c63ff', alpha=0.85, edgecolor='#2a2d3a'
)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel('Feature Importance Score')
ax.set_title(f'Top {top_n} Feature Importances — Random Forest', fontsize=13, fontweight='bold', pad=12)
ax.grid(True, axis='x', alpha=0.15)
plt.tight_layout()
plt.savefig('../visuals/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Show which numeric features appear in top 20
print('\nNumeric features in top 20:')
for i, name in enumerate(top_names):
    if name in NUMERIC_FEATURES:
        print(f'  Rank {i+1}: {name}  (importance={top_values[i]:.5f})')

## 14. Save Best Model & Metrics

In [ ]:
# Save the best model pipeline
model_path   = '../models/best_model.pkl'
metrics_path = '../models/metrics.json'

joblib.dump(best_pipe, model_path)

# Format metrics keys for the FastAPI /metrics endpoint
metrics_export = {
    'best_model': best_name,
    'models': {
        name.lower().replace(' ', '_'): v
        for name, v in all_metrics.items()
    }
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f'✅  Model saved  → {model_path}')
print(f'✅  Metrics saved → {metrics_path}')
print(f'\n🏆  Best model : {best_name}')
print(f'   Accuracy   : {all_metrics[best_name]["accuracy"]:.4f}')
print(f'   Precision  : {all_metrics[best_name]["precision"]:.4f}')
print(f'   Recall     : {all_metrics[best_name]["recall"]:.4f}')
print(f'   F1 Score   : {all_metrics[best_name]["f1"]:.4f}')

## 15. Summary of Visuals Generated

All charts have been saved to the `visuals/` directory:

| File | Description |
|---|---|
| `class_distribution.png` | Bar chart — Fake vs Real count |
| `rating_distribution.png` | Star rating histograms per class |
| `review_length_distribution.png` | Review length overlapping histogram |
| `feature_distributions.png` | Sentiment score and rating gap by label |
| `wordclouds.png` | Word clouds — Fake and Real side by side |
| `confusion_matrices.png` | 2×2 grid of confusion matrices |
| `roc_curves.png` | All 4 ROC curves on one chart |
| `model_comparison.png` | Grouped bar chart — Accuracy/Precision/Recall/F1 |
| `feature_importance.png` | Top 20 Random Forest features |

---

## 16. Discussion

*(Fill this in yourself based on your actual results — this is what the examiner will look for)*

**Questions to address in your report:**
1. Which model achieved the highest F1 score? Why might that be?
2. Which features appeared in the top 20 importances from Random Forest?
3. What does a high `rating_sentiment_gap` tell us about a review?
4. Why did we use `ComplementNB` instead of `MultinomialNB`?
5. What are the limitations of TF-IDF vs. BERT for this task?

---
*ReviewGuard — ML Course Project*